# Data Modeling — Dataset to Qdrant

1. Dataset preparation
2. Dense items → `Amazon-items-collection-01`
3. Hybrid items → `Amazon-items-collection-01-hybrid-search`
4. Reviews → `Amazon-reviews-collection-01`


## 0. Setup


In [ ]:
import json
import os
import gzip

import openai
import pandas as pd
import tiktoken

from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance,
    VectorParams,
    SparseVectorParams,
    Modifier,
    PayloadSchemaType,
    PointStruct,
    Document,
)

QDRANT_URL = os.environ["QDRANT_URL"]
EMBEDDING_MODEL = "text-embedding-3-small"
VECTOR_SIZE = 1536

qdrant_client = QdrantClient(url=QDRANT_URL)


def get_embedding(text, model=EMBEDDING_MODEL):
    """Embed a single string with OpenAI."""
    response = openai.embeddings.create(input=text, model=model)
    return response.data[0].embedding


def get_embeddings_batch(text_list, model=EMBEDDING_MODEL, batch_size=100):
    """Embed many strings in batches to stay within API limits."""
    if len(text_list) <= batch_size:
        response = openai.embeddings.create(input=text_list, model=model)
        return [embedding.embedding for embedding in response.data]

    all_embeddings = []
    counter = 1
    for i in range(0, len(text_list), batch_size):
        batch = text_list[i : i + batch_size]
        response = openai.embeddings.create(input=batch, model=model)
        all_embeddings.extend([embedding.embedding for embedding in response.data])
        print(f"Processed {counter * batch_size} of {len(text_list)}")
        counter += 1

    return all_embeddings


## 1. Dataset preparation


### 1.1 Inspect raw metadata


In [ ]:
with gzip.open("../../data/meta_Electronics.jsonl.gz", "rt") as f:
    first_line = json.loads(f.readline())

first_line


### 1.2 Filter by date (2022+)


In [ ]:
def filter_data_by_date(data: dict) -> bool:
    """Return True if the item should be filtered OUT (older than 2022)."""
    filter_out = False
    if int(data["details"]["Date First Available"][-4:]) < 2022:
        filter_out = True
    return filter_out


In [ ]:
with gzip.open("../../data/meta_Electronics.jsonl.gz", "rt") as fp:
    with open("../../data/meta_Electronics_2022_2023.jsonl", "a", encoding="utf-8") as fp_out:
        with open("../../data/meta_Electronics_2022_2023_no_date.jsonl", "a", encoding="utf-8") as fp_out_no_date:
            i = 0
            for line in fp:
                data = json.loads(line.strip())
                try:
                    if not filter_data_by_date(data):
                        json.dump(data, fp_out)
                        fp_out.write("\n")
                        fp_out.flush()
                except Exception:
                    json.dump(data, fp_out_no_date)
                    fp_out_no_date.write("\n")
                    fp_out_no_date.flush()
                i += 1
                if i % 10000 == 0:
                    print(f"Processed {i} lines")


### 1.3 Filter by main category


In [ ]:
def filter_category(data: dict) -> bool:
    """Return True if the item should be filtered OUT (no main_category)."""
    return data["main_category"] is None


In [ ]:
with open("../../data/meta_Electronics_2022_2023.jsonl", "r") as fp:
    with open("../../data/meta_Electronics_2022_2023_with_category.jsonl", "a", encoding="utf-8") as fp_out:
        with open("../../data/meta_Electronics_2022_2023_no_category.jsonl", "a", encoding="utf-8") as fp_out_no_category:
            i = 0
            for line in fp:
                data = json.loads(line.strip())
                if not filter_category(data):
                    json.dump(data, fp_out)
                    fp_out.write("\n")
                    fp_out.flush()
                else:
                    json.dump(data, fp_out_no_category)
                    fp_out_no_category.write("\n")
                    fp_out_no_category.flush()
                i += 1
                if i % 10000 == 0:
                    print(f"Processed {i} lines")


### 1.4 Filter by ratings and sample 1000


In [ ]:
df = pd.read_json("../../data/meta_Electronics_2022_2023_with_category.jsonl", lines=True)
df_ratings_100 = df[df["rating_number"] >= 100]
df_sample_1000 = df_ratings_100.sample(n=1000, random_state=42)

print(len(df), len(df_ratings_100), len(df_sample_1000))

df_ratings_100.to_json(
    "../../data/meta_Electronics_2022_2023_with_category_ratings_100.jsonl",
    orient="records",
    lines=True,
)
df_sample_1000.to_json(
    "../../data/meta_Electronics_2022_2023_with_category_ratings_100_sample_1000.jsonl",
    orient="records",
    lines=True,
)


### 1.5 Extract reviews for sampled ASINs


In [ ]:
df_sample_1000 = pd.read_json(
    "../../data/meta_Electronics_2022_2023_with_category_ratings_100_sample_1000.jsonl",
    lines=True,
)

with gzip.open("../../data/Electronics.jsonl.gz", "r") as fp:
    with open(
        "../../data/Electornics_2022_2023_with_category_ratings_100_sample_1000.jsonl",
        "a",
    ) as fp_out:
        id_list = set(df_sample_1000["parent_asin"].values)
        i = 0
        for line in fp:
            data = json.loads(line.strip())
            if data["parent_asin"] in id_list:
                json.dump(data, fp_out)
                fp_out.write("\n")
                fp_out.flush()
            i += 1
            if i % 100000 == 0:
                print(f"Processed {i} lines")


## 2. Dense items → `Amazon-items-collection-01`


### 2.1 Preprocess and sample 50


In [ ]:
def preprocess_description(row):
    return f"{row['title']} {' '.join(row['features'])}"


def extract_first_large_image(row):
    return row["images"][0].get("large", "")


df_items = pd.read_json(
    "../../data/meta_Electronics_2022_2023_with_category_ratings_100_sample_1000.jsonl",
    lines=True,
)
df_items["preprocessed_description"] = df_items.apply(preprocess_description, axis=1)
df_items["image"] = df_items.apply(extract_first_large_image, axis=1)

df_sample = df_items.sample(n=50, random_state=42)
df_data_to_embed = df_sample[
    ["preprocessed_description", "image", "rating_number", "price", "average_rating", "parent_asin"]
]
data_to_embed_dense = df_data_to_embed.to_dict(orient="records")
len(data_to_embed_dense), df_data_to_embed.head()


### 2.2 Create collection


In [ ]:
qdrant_client.create_collection(
    collection_name="Amazon-items-collection-01",
    vectors_config=VectorParams(size=VECTOR_SIZE, distance=Distance.COSINE),
)


### 2.3 Embed and upsert


In [ ]:
pointstructs_dense = []
for i, data in enumerate(data_to_embed_dense):
    embedding = get_embedding(data["preprocessed_description"])
    pointstructs_dense.append(
        PointStruct(
            id=i,
            vector=embedding,
            payload=data,
        )
    )

qdrant_client.upsert(
    collection_name="Amazon-items-collection-01",
    points=pointstructs_dense,
)

qdrant_client.get_collection("Amazon-items-collection-01")


## 3. Hybrid items → `Amazon-items-collection-01-hybrid-search`


### 3.1 Create collection and payload index


In [ ]:
qdrant_client.create_collection(
    collection_name="Amazon-items-collection-01-hybrid-search",
    vectors_config={
        "text-embedding-3-small": VectorParams(size=VECTOR_SIZE, distance=Distance.COSINE)
    },
    sparse_vectors_config={
        "bm25": SparseVectorParams(modifier=Modifier.IDF)
    },
)

qdrant_client.create_payload_index(
    collection_name="Amazon-items-collection-01-hybrid-search",
    field_name="parent_asin",
    field_schema=PayloadSchemaType.KEYWORD,
)


### 3.2 Preprocess and embed (batch)


In [ ]:
df_items_hybrid = pd.read_json(
    "../../data/meta_Electronics_2022_2023_with_category_ratings_100_sample_1000.jsonl",
    lines=True,
)
df_items_hybrid["preprocessed_description"] = df_items_hybrid.apply(preprocess_description, axis=1)
df_items_hybrid["image"] = df_items_hybrid.apply(extract_first_large_image, axis=1)

df_data_to_embed_hybrid = df_items_hybrid[
    ["preprocessed_description", "image", "rating_number", "price", "average_rating", "parent_asin"]
]
data_to_embed_hybrid = df_data_to_embed_hybrid.to_dict(orient="records")
text_to_embed_hybrid = [item["preprocessed_description"] for item in data_to_embed_hybrid]
embeddings_hybrid = get_embeddings_batch(text_to_embed_hybrid)

len(data_to_embed_hybrid), len(embeddings_hybrid)


### 3.3 Build points and upsert


In [ ]:
pointstructs_hybrid = []
i = 1
for embedding, data in zip(embeddings_hybrid, data_to_embed_hybrid):
    pointstructs_hybrid.append(
        PointStruct(
            id=i,
            vector={
                "text-embedding-3-small": embedding,
                "bm25": Document(
                    text=data["preprocessed_description"],
                    model="qdrant/bm25",
                ),
            },
            payload=data,
        )
    )
    i += 1

qdrant_client.upsert(
    collection_name="Amazon-items-collection-01-hybrid-search",
    points=pointstructs_hybrid[0:500],
    wait=True,
)
qdrant_client.upsert(
    collection_name="Amazon-items-collection-01-hybrid-search",
    points=pointstructs_hybrid[500:],
    wait=True,
)

qdrant_client.get_collection("Amazon-items-collection-01-hybrid-search")


## 4. Reviews → `Amazon-reviews-collection-01`


### 4.1 Create collection and payload index


In [ ]:
qdrant_client.create_collection(
    collection_name="Amazon-reviews-collection-01",
    vectors_config={
        "text-embedding-3-small": VectorParams(size=VECTOR_SIZE, distance=Distance.COSINE)
    },
)

qdrant_client.create_payload_index(
    collection_name="Amazon-reviews-collection-01",
    field_name="parent_asin",
    field_schema=PayloadSchemaType.KEYWORD,
)


### 4.2 Preprocess, filter tokens, embed


In [ ]:
def preprocess_reviews_data(row):
    return f"{row['title']} {row['text']}"


def count_tokens(row):
    encoding = tiktoken.encoding_for_model(EMBEDDING_MODEL)
    return len(encoding.encode(row["preprocessed_data"]))


df_reviews = pd.read_json(
    "../../data/Electornics_2022_2023_with_category_ratings_100_sample_1000.jsonl",
    lines=True,
)
df_reviews["preprocessed_data"] = df_reviews.apply(preprocess_reviews_data, axis=1)
df_reviews["token_count"] = df_reviews.apply(count_tokens, axis=1)
df_reviews = df_reviews[df_reviews["token_count"] < 8192]

print(len(df_reviews), df_reviews["token_count"].sum())

df_data_to_embed_reviews = df_reviews[["preprocessed_data", "parent_asin"]]
data_to_embed_reviews = df_data_to_embed_reviews.to_dict(orient="records")
text_to_embed_reviews = [item["preprocessed_data"] for item in data_to_embed_reviews]
embeddings_reviews = get_embeddings_batch(text_to_embed_reviews, batch_size=500)

len(data_to_embed_reviews), len(embeddings_reviews)


### 4.3 Build points and upsert


In [ ]:
pointstructs_reviews = []
i = 1
for embedding, data in zip(embeddings_reviews, data_to_embed_reviews):
    pointstructs_reviews.append(
        PointStruct(
            id=i,
            vector={"text-embedding-3-small": embedding},
            payload=data,
        )
    )
    i += 1

batch_size_qdrant = 100
counter = 1
for i in range(0, len(pointstructs_reviews), batch_size_qdrant):
    batch = pointstructs_reviews[i : i + batch_size_qdrant]
    qdrant_client.upsert(
        collection_name="Amazon-reviews-collection-01",
        points=batch,
        wait=True,
    )
    print(f"Processed {counter * batch_size_qdrant} of {len(pointstructs_reviews)}")
    counter += 1

qdrant_client.get_collection("Amazon-reviews-collection-01")


## 5. Collections created

- `Amazon-items-collection-01`
- `Amazon-items-collection-01-hybrid-search`
- `Amazon-reviews-collection-01`
